# Controlled RAG Evaluation — Results Walkthrough

A lightweight walkthrough of the **completed frozen experiment**. This notebook reads only the committed summary CSV files, so it can be reviewed without downloading the 15 source documents or loading the language models.

### Headline result
Basic RAG increased held-out factual accuracy from **34.4% to 70.0%** and reduced claim-level hallucination from **40.5% to 7.6%**. Enhanced RAG improved retrieval recall further, reaching **73.3% factual accuracy**, but its answer-quality gain over Basic RAG was not statistically significant.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root():
    here = Path.cwd()
    candidates = [here, here.parent, Path("/content/controlled-rag-university-documents")]
    for candidate in candidates:
        if (candidate / "results" / "heldout_system_summary.csv").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from the cloned repository, or update REPO_ROOT.")

REPO_ROOT = find_repo_root()
RESULTS = REPO_ROOT / "results"
CONFIG = REPO_ROOT / "config"
print("Repository root:", REPO_ROOT)


## 1. Held-out system comparison

In [ ]:
systems = pd.read_csv(RESULTS / "heldout_system_summary.csv")
display(systems[[
    "system_name", "factual_accuracy", "mean_completeness",
    "corpus_support_rate", "hallucination_rate",
    "correct_refusal_rate", "median_latency_seconds"
]].sort_values("factual_accuracy", ascending=False))


In [ ]:
order = ["LLM_ONLY", "BASIC_RAG", "ENHANCED_RAG"]
plot_df = systems.set_index("system_name").loc[order].reset_index()
labels = {"LLM_ONLY":"LLM-only","BASIC_RAG":"Basic RAG","ENHANCED_RAG":"Enhanced RAG"}

plt.figure(figsize=(7,4.5))
bars = plt.bar(plot_df["system_name"].map(labels), plot_df["factual_accuracy"]*100)
plt.ylabel("Factual accuracy (%)")
plt.ylim(0,100)
plt.title("Held-out factual accuracy")
for bar in bars:
    plt.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f"{bar.get_height():.1f}%", ha="center")
plt.show()


## 2. Retrieval configuration selection

In [ ]:
dev = pd.read_csv(RESULTS / "development_retrieval_summary.csv")
display(dev.sort_values(
    ["mean_recall_at_5","mean_mrr","mean_hit_rate_at_5","median_latency_seconds"],
    ascending=[False,False,False,True]
))

selection = json.loads((RESULTS / "development_selection.json").read_text())
print("Selected Enhanced configuration:")
print(json.dumps(selection["enhanced_config"], indent=2))


## 3. Paired statistical tests

In [ ]:
tests = pd.read_csv(RESULTS / "primary_statistical_tests.csv")
display(tests[[
    "outcome","test","system_a","system_b","p_value",
    "difference_b_minus_a","ci95_low","ci95_high","holm_adjusted_p"
]])


The pre-specified LLM-only → Basic RAG gains in factual accuracy and completeness remained significant after Holm correction. The Basic RAG → Enhanced RAG differences did not.


## 4. Retrieval success and answer quality (RQ3)

In [ ]:
odds = pd.read_csv(RESULTS / "rq3_odds_ratios.csv")
spearman = pd.read_csv(RESULTS / "rq3_mrr_completeness_spearman.csv")
display(odds)
display(spearman)


For **Basic RAG**, complete evidence retrieval was associated with higher factual accuracy (odds ratio ≈ **5.43**, Fisher p ≈ **0.007**) and MRR correlated positively with completeness (ρ ≈ **0.319**, p ≈ **0.008**). Neither relationship was statistically supported for Enhanced RAG.

This is an **association**, not an independent causal effect.


## 5. Frozen protocol

In [ ]:
protocol = json.loads((CONFIG / "frozen_protocol_summary.json").read_text())
print(json.dumps(protocol, indent=2))
